In [0]:
%sql
use catalog 0725catalog;
drop table if exists 0725catalog.bronze.customers_raw;
CREATE TABLE IF NOT EXISTS 0725catalog.bronze.customers_raw (
  customer_id STRING,
  country STRING
) USING DELTA;

truncate table 0725catalog.bronze.customers_raw;
-- SET spark.databricks.delta.retentionDurationCheck.enabled = false;
-- VACUUM 0725catalog.bronze.customers_raw RETAIN 0 HOURS;


### why cannot use copy into here 
COPY INTO bronze.customer_raw \
FROM '/Volumes/0725catalog/src/data' \
FILEFORMAT = CSV \
FORMAT_OPTIONS ('header' = 'true');

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
spark = SparkSession.builder.getOrCreate()

# Path to your managed volume
src_path = "/Volumes/0725catalog/src/data"

# Read CSV from volume (simulating Auto Loader)
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .load(src_path)
    .withColumn("ingest_time", current_timestamp())  # optional tracking column
)

# Append to Bronze Delta table
df.write.format("delta").mode("append").saveAsTable("bronze.customer_raw")


In [0]:
import json
file_path = "/Volumes/0725catalog/src/data/processed_files.json"

# Overwrite with an empty list
with open(file_path, "w") as f:
    json.dump([], f)

print("Tracking file reset. All files will be treated as new.")

In [0]:
import json
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
spark = SparkSession.builder.getOrCreate()

# Path to your managed volume
src_path = "/Volumes/0725catalog/src/data"

file_path = "/Volumes/0725catalog/src/data/processed_files.json"

# List files in volume folder
all_files = [f.path for f in dbutils.fs.ls("/Volumes/0725catalog/src/data")]
all_files = [f for f in all_files if not f.endswith(".json")]

# Check if tracking file exists and load processed files
if os.path.exists(file_path):
    with open(file_path, "r") as f:
        processed_files = json.load(f)
else:
    processed_files = []

print("Already processed files:")
print(processed_files)

# Filter to only new (unprocessed) files
new_files = [f for f in all_files if f not in processed_files]

if new_files:
    print("New files to process:")
    print(new_files)

    # Load and process new files
    #df = spark.read.format("csv").option("header", "true").load(new_files)
    
    # TODO: Add your processing logic here
    # e.g., df.write.format(...).save(...)
    # Read CSV from volume (simulating Auto Loader)
    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .load(new_files)
        .withColumn("ingest_time", current_timestamp())  # optional tracking column
)

    # Append to Bronze Delta table
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("bronze.customers_raw")
    # Update tracking list with newly processed files
    processed_files.extend(new_files)

    # Save updated list
    with open(file_path, "w") as f:
        json.dump(processed_files, f)

    print("Updated processed files list:")
    print(processed_files)
else:
    print("No new files to process.")

#### In paid verion, autoloader will handle the trick


In [0]:
from pyspark.sql.functions import input_file_name

# Define source and checkpoint paths
source_path = "/Volumes/0725catalog/src/data"
checkpoint_path = "/Volumes/0725catalog/src/checkpoints/autoloader"

# Read new files using Auto Loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")  # Specify file format
    .option("header", "true")            # CSV header option
    .load(source_path)
)

# Optional: Add filename column for tracking
df = df.withColumn("source_file", input_file_name())

# TODO: Add your processing logic here
# For example, write to a Delta table
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .start("/Volumes/0725catalog/output/data")
)